In [26]:
# pip install -U langchain-openai

In [27]:
# pip install -U "langchain[openai]"

In [28]:
import time
import re
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
# 1. Connect DB
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "agent_pm"

DATABASE_URL = (
    f"postgresql+psycopg2://"
    f"{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

db = SQLDatabase.from_uri(
    DATABASE_URL,
    sample_rows_in_table_info=3,
)

print(f"Connected to database: {DB_NAME}")
print(f"Database dialect: {db.dialect}")

schema_cache = db.get_table_info()
dialect = db.dialect
top_k = 5

Connected to database: agent_pm
Database dialect: postgresql


In [29]:
def clean_sql(text: str) -> str:
    text = text.strip()

    # remove ```sql ... ```
    text = re.sub(r"^```sql", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    return text

In [30]:
def is_safe_sql(sql: str) -> bool:
    sql_clean = sql.strip()

    # Cho phép bắt đầu bằng SELECT hoặc WITH (CTE)
    if not re.match(r"^\s*(SELECT|WITH)\b", sql_clean, flags=re.IGNORECASE):
        return False

    # Bắt buộc kết thúc bằng ;
    if not sql_clean.endswith(";"):
        return False

    # Cấm nhiều statement
    if sql_clean.count(";") != 1:
        return False

    # Cấm keyword ghi/xóa/sửa schema
    forbidden = [
        "INSERT", "UPDATE", "DELETE", "DROP", "ALTER",
        "TRUNCATE", "CREATE", "REPLACE", "MERGE",
        "GRANT", "REVOKE", "VACUUM", "CALL", "DO"
    ]

    pattern = r"\b(" + "|".join(forbidden) + r")\b"

    if re.search(pattern, sql_clean, flags=re.IGNORECASE):
        return False

    return True

In [ ]:

llm = ChatOpenAI(
    model="claude-haiku-4-5-20251001",
    api_key="fe_oa_b963e1daed256e2c821329dd1bac985e2e592148e0fcdbfc", 
    base_url="https://api.freemodel.dev/v1",
    streaming=True,
)

In [32]:
ans = llm.invoke("thủ đô của việt nam là gì?")
print(ans)

content='Thủ đô của Việt Nam là Hà Nội.' additional_kwargs={} response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-5.4', 'model_provider': 'openai'} id='lc_run--019e886f-cbab-7731-82fe-4e90e9ea8914' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 3307, 'output_tokens': 37, 'total_tokens': 3344, 'input_token_details': {'cache_read': 2816}, 'output_token_details': {'reasoning': 21}}


In [33]:
SCHEMA_COMPACT = """
Tables:
companies(id,name,code,currency_id)
currencies(id,code,symbol,rate)
users(id,full_name,email,role,department,position,company_id,company_name,active,is_super_admin)
projects(id,name,code,status,priority,start_date,end_date,description,total_hours,task_count,member_count,worklog_count,scope_count,milestone_count,customer_name,company_id,owner_id,account_manager_id,currency_id)
members(id,project_id,user_id,role,joined_at)
milestones(id,project_id,name,status,due_date,description,completion_pct,task_count,done_count)
tasks(id,name,status,priority,deadline,end_at,description,issues,result,total_hours,project_id,company_id,assignee_id,milestone_id,currency_id)
task_blockers(id,task_id,severity,description,resolved_at)
scopes(id,project_id,task_id,assignee_id,sequence,name,notes,estimated_hours,currency_id)
worklogs(id,work_date,description,hours,task_id,project_id,company_id,user_id,source,slot)
backlogs(id,status,source,work_date,description,hours,task_id,project_id,company_id,user_id,currency_id,approver_id,approved_at,rejected_reason)
meetings(id,company_id,project_id,title,held_at,summary,decisions,participants,created_by_id)
meeting_action_items(id,meeting_id,title,description,owner_name,owner_user_id,due_date,priority,status,created_task_id)
agent_follow_ups(id,task_id,user_id,channel,thread_id,question,status,asked_at,replied_at,reply_text)

Relations:
users.company_id=companies.id
companies.currency_id=currencies.id
projects.company_id=companies.id
projects.currency_id=currencies.id
projects.owner_id=users.id
projects.account_manager_id=users.id
tasks.company_id=companies.id
tasks.currency_id=currencies.id
worklogs.company_id=companies.id
backlogs.company_id=companies.id
tasks.project_id=projects.id
milestones.project_id=projects.id
members.project_id=projects.id
scopes.project_id=projects.id
worklogs.project_id=projects.id
backlogs.project_id=projects.id
meetings.project_id=projects.id
meetings.company_id=companies.id
tasks.assignee_id=users.id
scopes.assignee_id=users.id
worklogs.user_id=users.id
backlogs.user_id=users.id
backlogs.approver_id=users.id
members.user_id=users.id
meetings.created_by_id=users.id
worklogs.task_id=tasks.id
backlogs.task_id=tasks.id
scopes.task_id=tasks.id
task_blockers.task_id=tasks.id
agent_follow_ups.task_id=tasks.id
agent_follow_ups.user_id=users.id
tasks.milestone_id=milestones.id
meeting_action_items.meeting_id=meetings.id
meeting_action_items.owner_user_id=users.id
meeting_action_items.created_task_id=tasks.id

Enums:
projects.status: PLANNED, PENDING, IN_PROGRESS, DONE, CANCELLED
tasks.status: TODO, IN_PROGRESS, DONE, CANCELLED
backlogs.status: PENDING, APPROVED, REJECTED
backlogs.source: manual, checkin, import
task_blockers.severity: LOW, MED, HIGH, CRITICAL
meeting_action_items.status: DRAFT, APPROVED, REJECTED
agent_follow_ups.status: PENDING, REPLIED, EXPIRED
priority: LOW, MEDIUM, HIGH, URGENT
users.role: ADMIN, MANAGER, MEMBER, VIEWER, SUPER_ADMIN

Rules:
running_project = projects.status='IN_PROGRESS'
pending_project = projects.status='PENDING'
cancelled_project = projects.status='CANCELLED'
done_project = projects.status='DONE'
active_project = projects.status NOT IN ('DONE','CANCELLED')
done_task = tasks.status='DONE'
remaining_task = tasks.status<>'DONE'
todo_task = tasks.status='TODO'
cancelled_task = tasks.status='CANCELLED'
overdue_task = tasks.deadline<CURRENT_DATE AND tasks.status<>'DONE'
blocked_task = EXISTS task_blockers WHERE task_id=tasks.id AND resolved_at IS NULL
milestone_progress = milestones.completion_pct
project_progress = DONE tasks / total tasks
project_hours = projects.total_hours
approved_backlog_hours = SUM(backlogs.hours) WHERE backlogs.status='APPROVED'
pending_backlog = backlogs.status='PENDING'
open_action_item = meeting_action_items.status='DRAFT'
pending_follow_up = agent_follow_ups.status='PENDING'
cost/budget fields were removed; do not query budget,total_cost,budget_remaining,estimated_total_cost,estimated_cost,estimated_rate,cost_per_hour_snapshot,total_cost_snapshot,member_rates,estimated_total_hours.
"""


In [34]:
SYSTEM_PROMPT = f"""Bạn là một trợ lý AI chuyên nghiệp giúp chuyển đổi các câu hỏi liên quan đến dự án, 
task thành các câu truy vấn SQL để truy xuất dữ liệu từ cơ sở dữ liệu quản lý dự án. Bạn sẽ nhận được một câu hỏi 
từ người dùng và cần tạo ra một câu truy vấn SQL chính xác, an toàn và hiệu quả để trả lời câu hỏi đó dựa trên schema của cơ sở dữ liệu đã được cung cấp.

Dưới đây là schema:
{SCHEMA_COMPACT}
Lưu ý quan trọng:
- Chỉ tạo câu truy vấn SQL sử dụng các bảng và cột đã được cung cấp
- Câu truy vấn phải bắt đầu bằng SELECT hoặc WITH và kết thúc bằng dấu chấm phẩy (;)
- Không được phép tạo câu truy vấn có chứa các lệnh nguy hiểm như INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE, CREATE, REPLACE, MERGE, GRANT, REVOKE, VACUUM, CALL, DO
- Câu truy vấn phải trả về dữ liệu phù hợp để trả lời câu hỏi, không được trả về dữ liệu thừa hoặc thiếu
- Cố gắng tối ưu câu truy vấn để trả về kết quả nhanh nhất có thể, tránh sử dụng các phép toán phức tạp hoặc subquery không cần thiết
- Viết câu truy vấn bằng tiếng Việt nếu có thể, nhưng vẫn phải tuân thủ cú pháp SQL chuẩn
- Nếu câu hỏi không thể trả lời bằng SQL, hãy trả về một câu truy vấn đơn giản trả về một thông báo phù hợp, ví dụ: SELECT 'Câu hỏi không thể trả lời bằng SQL' AS message;
"""

In [35]:
async def generate_sql(question: str, company_id: int = 1) -> str:
    # Thay placeholder theo thứ tự: top_k, company_id rồi mới đến schema_cache
    # (schema_cache có thể chứa ký tự `{}` từ sample row → để cuối cho an toàn).
    prompt = (
        SYSTEM_PROMPT
        .replace("{top_k}", str(top_k))
        .replace("{company_id}", str(company_id))
        .replace("{schema_cache}", schema_cache)
    )

    start = time.perf_counter()

    response = await llm.ainvoke([
        SystemMessage(content=prompt),
        HumanMessage(content=question),
    ])

    elapsed = time.perf_counter() - start

    sql = clean_sql(response.content)

    print("=== GENERATED SQL ===")
    print(sql)
    print(f"Generate SQL time: {elapsed:.3f}s")

    if not is_safe_sql(sql):
        raise ValueError(f"Unsafe SQL generated:\n{sql}")

    return sql

In [36]:
# Test thời gian token đầu tiên (TTFT) với streaming

question = "Liệt kê task quá hạn"
company_id = 1

prompt = (
    SYSTEM_PROMPT
    .replace("{top_k}", str(top_k))
    .replace("{company_id}", str(company_id))
    .replace("{schema_cache}", schema_cache)
)

messages = [
    SystemMessage(content=prompt),
    HumanMessage(content=question),
]

start = time.perf_counter()

first_token = None

async for chunk in llm.astream(messages):
    if first_token is None:
        first_token = time.perf_counter()
        print(
            f"TTFT: {first_token - start:.3f}s"
        )

print(
    f"Total: {time.perf_counter() - start:.3f}s"
)


TTFT: 4.064s
Total: 5.888s


In [37]:
async def ask_db(question: str, company_id: int = 1):
    sql = await generate_sql(question, company_id=company_id)

    start = time.perf_counter()

    result = db.run(sql)

    elapsed = time.perf_counter() - start

    print("\n=== QUERY RESULT ===")
    print(result)
    print(f"DB execution time: {elapsed:.3f}s")

    return {
        "question": question,
        "company_id": company_id,
        "sql": sql,
        "result": result,
    }

In [38]:
await ask_db("""Liệt kê task quá hạn""")




=== GENERATED SQL ===
SELECT
  t.id,
  t.name,
  t.status,
  t.priority,
  t.deadline,
  t.end_at,
  t.total_hours,
  p.id AS project_id,
  p.name AS project_name,
  u.id AS assignee_id,
  u.full_name AS assignee_name
FROM tasks t
LEFT JOIN projects p ON p.id = t.project_id
LEFT JOIN users u ON u.id = t.assignee_id
WHERE t.deadline < CURRENT_DATE
  AND t.status <> 'DONE'
ORDER BY t.deadline ASC, t.priority DESC, t.id ASC;
Generate SQL time: 8.942s

=== QUERY RESULT ===
[(20, 'Spike chat layout 3 cột', 'CANCELLED', 'LOW', datetime.date(2026, 5, 21), None, 4.0, 7, 'Experimental Chat UI', 2, 'Nguyễn Văn Admin'), (4, 'Kết nối AgentMessageRouter', 'IN_PROGRESS', 'URGENT', datetime.date(2026, 5, 29), None, 11.0, 1, 'Gapo Test CRM Rollout', 608678190, 'Đặng Trần Tấn Lực'), (10, 'Fallback route khi LLM lỗi', 'IN_PROGRESS', 'URGENT', datetime.date(2026, 5, 29), None, 10.0, 3, 'GapoWork Agent Integration', 2, 'Nguyễn Văn Admin'), (25, 'Test notification: điền worklog và cập nhật tiến độ', 'IN_PR

{'question': 'Liệt kê task quá hạn',
 'company_id': 1,
 'sql': "SELECT\n  t.id,\n  t.name,\n  t.status,\n  t.priority,\n  t.deadline,\n  t.end_at,\n  t.total_hours,\n  p.id AS project_id,\n  p.name AS project_name,\n  u.id AS assignee_id,\n  u.full_name AS assignee_name\nFROM tasks t\nLEFT JOIN projects p ON p.id = t.project_id\nLEFT JOIN users u ON u.id = t.assignee_id\nWHERE t.deadline < CURRENT_DATE\n  AND t.status <> 'DONE'\nORDER BY t.deadline ASC, t.priority DESC, t.id ASC;",
 'result': "[(20, 'Spike chat layout 3 cột', 'CANCELLED', 'LOW', datetime.date(2026, 5, 21), None, 4.0, 7, 'Experimental Chat UI', 2, 'Nguyễn Văn Admin'), (4, 'Kết nối AgentMessageRouter', 'IN_PROGRESS', 'URGENT', datetime.date(2026, 5, 29), None, 11.0, 1, 'Gapo Test CRM Rollout', 608678190, 'Đặng Trần Tấn Lực'), (10, 'Fallback route khi LLM lỗi', 'IN_PROGRESS', 'URGENT', datetime.date(2026, 5, 29), None, 10.0, 3, 'GapoWork Agent Integration', 2, 'Nguyễn Văn Admin'), (25, 'Test notification: điền worklog và 

In [39]:
print(db.get_usable_table_names())

['agent_audit_log', 'agent_follow_ups', 'agent_memory', 'automations', 'backlogs', 'channel_identities', 'checkin_sessions', 'companies', 'currencies', 'gapo_user_maps', 'meeting_action_items', 'meetings', 'members', 'milestones', 'projects', 'scopes', 'task_blockers', 'task_status', 'tasks', 'users', 'worklogs']
